In [ ]:
import shutil
from pathlib import Path
from typing import Any, Callable, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
from PIL import Image
from torchvision.datasets.vision import VisionDataset
import albumentations as A


class CustomImageDataset(VisionDataset):
    def __init__(
        self,
        root: Union[str, Path],
        target_size: tuple = (224, 224),
        train: bool = True,
        transform: Optional[Callable] = None,
        target_transform: Optional[Callable] = None,
    ):
        super().__init__(str(root), transform=transform, target_transform=target_transform)

        self.root = Path(root)
        self.target_size = target_size
        self.train = train

        self.data = self._load_data()

    def _load_data(self) -> pd.DataFrame:
        data = []
        phase = "train" if self.train else "test"
        image_root = self.root / phase

        # 建立 label 對應：{'1': 0, '2': 1, '3': 2, '4': 3}
        class_dirs = sorted([d for d in image_root.iterdir() if d.is_dir()])
        self.class_to_idx = {d.name: idx for idx, d in enumerate(class_dirs)}

        for class_dir in class_dirs:
            label = self.class_to_idx[class_dir.name]  # 轉為 0-based label
            for img_path in class_dir.glob("*.jpg"):
                relative_path = img_path.relative_to(self.root)
                data.append([relative_path.as_posix(), label])

        df = pd.DataFrame(data, columns=["path", "category"])
        return df

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, index: int) -> Tuple[Any, int]:
        row = self.data.iloc[index]
        img_path = self.root / row["path"]
        image = Image.open(img_path).convert("RGB")
    
        if self.transform:
            try:
                # 嘗試 Albumentations（需要 numpy 格式）
                image_np = np.array(image)
                augmented = self.transform(image=image_np)
                image = augmented["image"]
            except Exception:
                # 回退 torchvision transform（傳入 PIL.Image）
                image = self.transform(image)
    
        target = int(row["category"])
        if self.target_transform:
            target = self.target_transform(target)
    
        return image, target


    @property
    def targets(self) -> List[int]:
        return self.data["category"].tolist()

    @property
    def classes(self) -> List[int]:
        return sorted(self.data["category"].unique().tolist())

    def __str__(self) -> str:
        return f"CustomImageDataset(train={self.train}, size={len(self)})"


In [ ]:
from typing import Optional

import torch
from deprecated.sphinx import deprecated
from torch import Tensor
from torch.nn import CrossEntropyLoss, Module
from torch.nn.functional import one_hot


class CustomTargetsLoss(torch.nn.Module):
    """
    Base class for implementing a soft labelling loss using class-dependent target smoothing.

    This loss modifies the hard class labels by combining one-hot encoding with prior
    class probabilities. The result is a soft target distribution used as input to a
    base loss function that supports probabilistic targets (e.g., KL divergence or
    soft cross-entropy).

    The smoothing is controlled by the `eta` parameter, where `eta=0` corresponds to
    standard one-hot labels and `eta=1` corresponds to using only the prior class
    probabilities.

    Parameters
    ----------
    base_loss : torch.nn.Module
        The base loss function to apply between predictions and soft targets.
        It must accept `y_true` as a tensor of probabilities, not class indices.
        Specifically, `y_true` should be a vector of probabilities or a one-hot encoded
        vector, where each element represents the probability of the corresponding class

    cls_probs : torch.Tensor
        A tensor of shape (J, J), where each row `j` corresponds to a class-conditional
        target distribution for class `j`. This is used to create the soft targets.

    eta : float, default=1.0
        A scalar in [0, 1] controlling the degree of smoothing applied to the targets.
        Higher values increase the influence of the prior class distributions.

    Example
    -------
    >>> import torch
    >>> import torch.nn as nn
    >>> from dlordinal.losses import CustomTargetsLoss
    >>> base_loss_fn = nn.CrossEntropyLoss()
    >>> cls_probs = torch.tensor([[0.9, 0.075, 0.025], [0.1, 0.6, 0.3], [0.05, 0.15, 0.8]])
    >>> custom_loss_fn = CustomTargetsLoss(base_loss=base_loss_fn, cls_probs=cls_probs, eta=0.5)
    >>> y_pred = torch.randn(2, 3)
    >>> y_true = torch.tensor([0, 2])
    >>> loss = custom_loss_fn(y_pred, y_true)
    >>> print(loss)
    """

    def __init__(
        self,
        base_loss: Module,
        cls_probs: Tensor,
        eta: float = 1.0,
    ):
        super().__init__()

        self.base_loss = base_loss
        self.num_classes = cls_probs.size(0)
        self.eta = eta

        # Default class probs initialized to ones
        self.register_buffer("cls_probs", cls_probs.float())

    def forward(self, input: Tensor, target: Tensor) -> Tensor:
        """
        Computes the loss between the input predictions and the target labels.

        Parameters
        ----------
        input : torch.Tensor
            A float tensor of shape (N, J) containing predicted logits or probabilities,
            where N is the batch size and J is the number of classes. The expected format
            (logits vs probabilities) depends on the specific base loss function.

        target : torch.Tensor
            An integer tensor of shape (N,) containing the class indices (0 ≤ target < J)
            corresponding to the correct classes for each sample.

        Returns
        -------
        torch.Tensor
            A scalar tensor representing the computed loss.
            
        """
        device = input.device 

        y_prob = self.get_buffer("cls_probs")[target].to(device)
        target_oh = one_hot(target, self.num_classes).to(device)

        y_true = (1.0 - self.eta) * target_oh + self.eta * y_prob

        return self.base_loss(input, y_true)


# TODO: remove in 3.0.0
@deprecated(
    version="2.4.0",
    reason="Use CustomTargetsLoss instead with CrossEntropyLoss as base_loss. Will be removed in 3.0.0.",
    category=FutureWarning,
)
class CustomTargetsCrossEntropyLoss(CustomTargetsLoss):
    def __init__(
        self,
        cls_probs: Tensor,
        eta: float = 1.0,
        weight: Optional[Tensor] = None,
        size_average=None,
        ignore_index: int = -100,
        reduce=None,
        reduction: str = "mean",
        label_smoothing: float = 0.0,
    ):
        base_loss = CrossEntropyLoss(
            weight=weight,
            size_average=size_average,
            ignore_index=ignore_index,
            reduce=reduce,
            reduction=reduction,
        )
        super().__init__(
            base_loss=base_loss,
            cls_probs=cls_probs,
            eta=eta,
        )

In [ ]:
import numpy as np


def get_intervals(n):
    """Get n evenly-spaced intervals in :math:`[0,1]`.

    Parameters
    ----------
    n : int
        Number of intervals.

    Returns
    -------
    intervals: list
        List of intervals.
    """

    points = np.linspace(1e-9, 1 - 1e-9, n + 1)
    intervals = []
    for i in range(0, points.size - 1):
        intervals.append((points[i], points[i + 1]))

    return intervals


def triangular_cdf(x: float, a: float, b: float, c: float):
    """
    Triangular distribution CDF.

    Parameters
    ----------
    x : float
        Value.
    a : float
        Lower bound.
    b : float
        Upper bound.
    c : float
        Mode.
    """
    if x <= a:
        return 0
    elif a < x < c:
        return pow(x - a, 2) / ((b - a) * (c - a))
    elif c < x < b:
        return 1 - pow(b - x, 2) / ((b - a) * (b - c))
    else:  # b <= x
        return 1

In [ ]:
import math

import numpy as np

# from .utils import get_intervals, triangular_cdf


def get_triangular_soft_labels(J: int, alpha2: float = 0.01, verbose: int = 0):
    """
    Get soft labels using triangular distributions for ``J`` classes or splits using
    the approach described in :footcite:t:`vargas2023softlabelling`.
    The :math:`[0,1]` interval is split into ``J`` intervals and the probability for
    each interval is computed as the difference between the value of the triangular
    distribution function for the interval boundaries. The probability for the first
    interval is computed as the value of the triangular distribution function for the
    first interval boundary.

    The triangular distribution function is denoted as :math:`\\text{p}(x, a, b, c)`
    where :math:`a`, :math:`b` and :math:`c` are the parameters of the distribution,
    and are determined by the number of classes :math:`J` and the value of the
    :math:`\\alpha_2` parameter. The value of :math:`\\alpha_2` represents the
    probability that is assigned to the adjacent classes of the target class. The
    parameters :math:`a`, :math:`b` and :math:`c` for class :math:`j` are computed
    as follows:

    .. math::
        a_j = \\begin{cases}
            0 & \\text{if } j = 1 \\\\
            \\frac{2j - 2 - 4j\\alpha_2 + 2\\alpha_2 \\pm
              \\sqrt{2\\alpha_2}}{2n(1 - 2\\alpha_2)} & \\text{if } 1 < j < J\\\\
            1 + \\frac{1}{\\pm J (\\sqrt{\\alpha_3} - 1)} & \\text{if } j = J
        \\end{cases}

    .. math::
        b_j = \\begin{cases}
            \\frac{1}{(1 - \\sqrt{\\alpha_1})n} & \\text{if } j = 1 \\\\
            \\frac{2j - 4j\\alpha_2 + 2\\alpha_2 \\pm
             \\sqrt{2\\alpha_2}}{2n(1 - 2\\alpha_2)} & \\text{if } 1 < j < J\\\\
             1 & \\text{if } j = J
        \\end{cases}

    .. math::
        c_j = \\begin{cases}
            0 & \\text{if } j = 1 \\\\
            \\frac{a + b}{2} & \\text{if } 1 < j  < J\\\\
            1 & \\text{if } j = J
        \\end{cases}

    The value of :math:`\\alpha_1`, that represents the error for the first class,
    is computed as follows:

    .. math::
        \\alpha_1 = \\left(\\frac{1 - \\sqrt{1 - 4(1 - 2\\alpha_2)(2\\alpha_2 -
          \\sqrt{2\\alpha_2})}}{2}\\right)^2

    The value of :math:`\\alpha_3`, that represents the error for the last class,
    is computed as follows:

    .. math::
        \\alpha_3 = \\left(\\frac{1 -
        \\sqrt{1 - 4\\left(\\frac{J - 1}{J}\\right)^2(1 - 2\\alpha_2)(\\sqrt{2\\alpha_2}
          (-1 + \\sqrt{2\\alpha_2}))}}{2}\\right)^2


    The value of :math:`\\alpha_2` is given by the user.

    Parameters
    ----------
    J : int
        Number of classes or splits (:math:`J`).
    alpha2 : float, optional, default=0.01
        Value of the :math:`\\alpha_2` parameter.
    verbose : int, optional, default=0
        Verbosity level.

    Raises
    ------
    ValueError
        If ``J`` is not a positive integer greater than 1.
        If ``alpha2`` is not a float between 0 and 1.

    Returns
    -------
    probs : 2d array-like of shape (J, J)
        Matrix of probabilities where each row represents the true class
        and each column the probability for class j.

    Example
    -------
    >>> from dlordinal.soft_labelling import get_triangular_soft_labels
    >>> get_triangular_soft_labels(5)
    array([[0.98845494, 0.01154505, 0.        , 0.        , 0.        ],
           [0.01      , 0.98      , 0.01      , 0.        , 0.        ],
           [0.        , 0.01      , 0.98      , 0.01      , 0.        ],
           [0.        , 0.        , 0.01      , 0.98      , 0.01      ],
           [0.        , 0.        , 0.        , 0.00505524, 0.99494475]])
    """

    if J < 2 or not isinstance(J, int):
        raise ValueError(f"{J=} must be a positive integer greater than 1")

    if alpha2 < 0 or alpha2 > 1:
        raise ValueError(f"{alpha2=} must be a float between 0 and 1")

    if verbose >= 1:
        print(f"Computing triangular probabilities for {J=} and {alpha2=}...")

    def compute_alpha1(alpha2):
        c_minus = (1 - 2 * alpha2) * (2 * alpha2 - math.sqrt(2 * alpha2))

        return pow((1 - math.sqrt(1 - 4 * c_minus)) / 2, 2)

    def compute_alpha3(alpha2):
        c1 = (
            pow((J - 1) / J, 2)
            * (1 - 2 * alpha2)
            * (math.sqrt(2 * alpha2) * (-1 + math.sqrt(2 * alpha2)))
        )

        return pow((1 - math.sqrt(1 - 4 * c1)) / 2, 2)

    alpha1 = compute_alpha1(alpha2)
    alpha3 = compute_alpha3(alpha2)

    if verbose >= 1:
        print(f"{alpha1=}, {alpha2=}, {alpha3=}")

    def b1(J):
        return 1.0 / ((1.0 - math.sqrt(alpha1)) * J)

    def aj(J, j):
        num1 = 2.0 * j - 2 - 4 * j * alpha2 + 2 * alpha2
        num2 = math.sqrt(2 * alpha2)
        den = 2.0 * J * (1 - 2 * alpha2)

        max_value = (j - 1.0) / J

        # +-
        return (
            (num1 + num2) / den
            if (num1 + num2) / den < max_value
            else (num1 - num2) / den
        )

    def bj(J, j):
        num1 = 2.0 * j - 4 * j * alpha2 + 2 * alpha2
        num2 = math.sqrt(2 * alpha2)
        den = 2.0 * J * (1 - 2 * alpha2)

        min_value = j / J

        # +-
        return (
            (num1 + num2) / den
            if (num1 + num2) / den > min_value
            else (num1 - num2) / den
        )

    def aJ(J):
        aJ_plus = 1.0 + 1.0 / (J * (math.sqrt(alpha3) - 1.0))
        aJ_minus = 1.0 + 1.0 / (-J * (math.sqrt(alpha3) - 1.0))
        return aJ_plus if aJ_plus > 0.0 else aJ_minus

    if verbose >= 3:
        print(f"{b1(J)=}, {aJ(J)=}, {aj(J, 1)=}, {bj(J,1)=}")
        for i in range(1, J + 1):
            print(f"{i=}  {aj(J, i)=}, {bj(J,i)=}")

    intervals = get_intervals(J)
    probs = []

    # Compute probability for each interval (class) using the distribution function.
    for j in range(1, J + 1):
        j_probs = []
        if j == 1:
            a = 0.0
            b = b1(J)
            c = 0.0
        elif j == J:
            a = aJ(J)
            b = 1.0
            c = 1.0
        else:
            a = aj(J, j)
            b = bj(J, j)
            c = (a + b) / 2.0

        if verbose >= 1:
            print(f"Class: {j}, {a=}, {b=}, {c=}, (j-1)/J={(j-1)/J}, (j/J)={j/J}")

        for interval in intervals:
            j_probs.append(
                triangular_cdf(interval[1], a, b, c)
                - triangular_cdf(interval[0], a, b, c)
            )
            if verbose >= 2:
                print(f"\tinterval: {interval}, prob={j_probs[-1]}")

        probs.append(j_probs)

    return np.array(probs)

In [ ]:
from typing import Optional

import torch
from deprecated.sphinx import deprecated
from torch import Tensor
from torch.nn import CrossEntropyLoss, Module

# from dlordinal.soft_labelling import get_triangular_soft_labels

# from .custom_targets_loss import CustomTargetsLoss


class TriangularLoss(CustomTargetsLoss):
    """Triangular regularised loss from :footcite:t:`vargas2023softlabelling`.

    This loss function combines a base loss function (such as cross-entropy) with
    a triangular regularisation term, which distributes probabilities to adjacent
    classes. The parameter `alpha2` controls the amount of probability deposited
    into adjacent classes, and `eta` controls the strength of the regularisation.

    Parameters
    ----------
    base_loss : torch.nn.Module
        The base loss function (e.g., `CrossEntropyLoss`). It must accept `y_true`
        as a probability distribution (e.g., one-hot or soft labels).
    num_classes : int
        Number of classes. This defines the size of the probability distribution.
    alpha2 : float, default=0.05
        Parameter that controls the amount of probability deposited in adjacent classes.
        Higher values increase the contribution of adjacent classes.
    eta : float, default=1.0
        Regularisation parameter that controls the influence of the triangular regularisation
        term. A value of 1.0 gives equal weight to the base loss and the triangular term,
        while smaller values reduce the regularisation strength.

    Example
    -------
    >>> import torch
    >>> from dlordinal.losses import TriangularLoss
    >>> from torch.nn import CrossEntropyLoss
    >>> num_classes = 5
    >>> base_loss = CrossEntropyLoss()
    >>> loss = TriangularLoss(base_loss, num_classes)
    >>> input = torch.randn(3, num_classes)  # Predicted logits for 3 samples
    >>> target = torch.randint(0, num_classes, (3,))  # Ground truth class indices
    >>> output = loss(input, target)  # Compute the loss
    >>> print(output)
    """

    def __init__(
        self,
        base_loss: Module,
        num_classes: int,
        alpha2: float = 0.05,
        eta: float = 1.0,
    ):
        # Precompute class probabilities for each label
        cls_probs = torch.tensor(get_triangular_soft_labels(num_classes, alpha2))
        super().__init__(
            base_loss=base_loss,
            cls_probs=cls_probs,
            eta=eta,
        )

    forward = CustomTargetsLoss.forward


# TODO: remove in 3.0.0
@deprecated(
    version="2.4.0",
    reason="Use TriangularLoss instead with CrossEntropyLoss as base_loss. Will be removed in 3.0.0.",
    category=FutureWarning,
)
class TriangularCrossEntropyLoss(TriangularLoss):
    def __init__(
        self,
        num_classes: int,
        alpha2: float = 0.05,
        eta: float = 1.0,
        weight: Optional[Tensor] = None,
        size_average=None,
        ignore_index: int = -100,
        reduce=None,
        reduction: str = "mean",
    ):
        base_loss = CrossEntropyLoss(
            weight=weight,
            size_average=size_average,
            ignore_index=ignore_index,
            reduce=reduce,
            reduction=reduction,
        )
        super().__init__(
            base_loss=base_loss,
            num_classes=num_classes,
            alpha2=alpha2,
            eta=eta,
        )

In [ ]:
import json
import os
from pathlib import Path
from typing import Callable, Dict, Optional

import numpy as np
from sklearn.metrics import confusion_matrix, recall_score


def ranked_probability_score(y_true, y_proba):
    """Computes the ranked probability score as presented in :footcite:t:`janitza2016random`.

    Parameters
    ----------
    y_true : array-like
            Target labels.
    y_proba : array-like
            Predicted probabilities.

    Returns
    -------
    rps : float
            The ranked probability score.

    Examples
    --------
    >>> import numpy as np
    >>> from dlordinal.metrics import ranked_probability_score
    >>> y_true = np.array([0, 0, 3, 2])
    >>> y_pred = np.array([[0.2, 0.4, 0.2, 0.2], [0.7, 0.1, 0.1, 0.1], [0.5, 0.05, 0.1, 0.35], [0.1, 0.05, 0.65, 0.2]])
    >>> ranked_probability_score(y_true, y_pred)
    0.5068750000000001
    """
    y_true = np.array(y_true)
    y_proba = np.array(y_proba)

    y_oh = np.zeros(y_proba.shape)
    y_oh[np.arange(len(y_true)), y_true] = 1

    y_oh = y_oh.cumsum(axis=1)
    y_proba = y_proba.cumsum(axis=1)

    rps = 0
    for i in range(len(y_true)):
        if y_true[i] in np.arange(y_proba.shape[1]):
            rps += np.power(y_proba[i] - y_oh[i], 2).sum()
        else:
            rps += 1
    return rps / len(y_true)


def minimum_sensitivity(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Computes the sensitivity by class and returns the lowest value.

    Parameters
    ----------
    y_true : array-like
            Target labels.
    y_pred : array-like
            Predicted probabilities or labels.

    Returns
    -------
    ms : float
            Minimum sensitivity.

    Examples
    --------
    >>> import numpy as np
    >>> from dlordinal.metrics import minimum_sensitivity
    >>> y_true = np.array([0, 0, 1, 2, 3, 0, 0])
    >>> y_pred = np.array([0, 1, 1, 2, 3, 0, 1])
    >>> minimum_sensitivity(y_true, y_pred)
    0.5
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    if len(y_true.shape) > 1:
        y_true = np.argmax(y_true, axis=1)
    if len(y_pred.shape) > 1:
        y_pred = np.argmax(y_pred, axis=1)

    sensitivities = recall_score(y_true, y_pred, average=None)
    return np.min(sensitivities)


def accuracy_off1(y_true: np.ndarray, y_pred: np.ndarray, labels=None) -> float:
    """Computes the accuracy of the predictions, allowing errors if they occur in an
    adjacent class.

    Parameters
    ----------
    y_true : array-like
            Target labels.
    y_pred : array-like
            Predicted probabilities or labels.
    labels : array-like or None
            Labels of the classes. If None, the labels are inferred from the data.

    Returns
    -------
    acc : float
            1-off accuracy.

    Examples
    --------
    >>> import numpy as np
    >>> from dlordinal.metrics import accuracy_off1
    >>> y_true = np.array([0, 0, 1, 2, 3, 0, 0])
    >>> y_pred = np.array([0, 1, 1, 2, 0, 0, 1])
    >>> accuracy_off1(y_true, y_pred)
    0.8571428571428571
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    if len(y_true.shape) > 1:
        y_true = np.argmax(y_true, axis=1)
    if len(y_pred.shape) > 1:
        y_pred = np.argmax(y_pred, axis=1)
    if labels is None:
        labels = np.unique(y_true)

    conf_mat = confusion_matrix(y_true, y_pred, labels=labels)
    n = conf_mat.shape[0]
    mask = np.eye(n, n) + np.eye(n, n, k=1), +np.eye(n, n, k=-1)
    correct = mask * conf_mat

    return 1.0 * np.sum(correct) / np.sum(conf_mat)


def gmsec(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Geometric Mean of the Sensitivity of the Extreme Classes (GMSEC). It was proposed
    in (:footcite:t:`vargas2024improving`) with the aim of assessing the performance of
    the classification performance for the first and the last classes.

    Parameters
    ----------
    y_true : array-like
            Target labels.
    y_pred : array-like
            Predicted probabilities or labels.

    Returns
    -------
    gmec : float
            Geometric mean of the sensitivities of the extreme classes.

    Examples
    --------
    >>> import numpy as np
    >>> from dlordinal.metrics import gmsec
    >>> y_true = np.array([0, 0, 1, 2, 3, 0, 0])
    >>> y_pred = np.array([0, 1, 1, 2, 3, 0, 1])
    >>> gmsec(y_true, y_pred)
    0.7071067811865476
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    if len(y_true.shape) > 1:
        y_true = np.argmax(y_true, axis=1)
    if len(y_pred.shape) > 1:
        y_pred = np.argmax(y_pred, axis=1)

    sensitivities = recall_score(y_true, y_pred, average=None)
    return np.sqrt(sensitivities[0] * sensitivities[-1])


def amae(y_true: np.ndarray, y_pred: np.ndarray):
    """Computes the average mean absolute error computed independently for each class
    as presented in :footcite:t:`baccianella2009evaluation`.

    Parameters
    ----------
    y_true : array-like
            Targets labels with one-hot or integer encoding.
    y_pred : array-like
            Predicted probabilities or labels.

    Returns
    -------
    amae : float
            Average mean absolute error.

    Examples
    --------
    >>> import numpy as np
    >>> from dlordinal.metrics import amae
    >>> y_true = np.array([0, 0, 1, 2, 3, 0, 0])
    >>> y_pred = np.array([0, 1, 1, 2, 3, 0, 1])
    >>> amae(y_true, y_pred)
    0.125
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    if len(y_true.shape) > 1:
        y_true = np.argmax(y_true, axis=1)
    if len(y_pred.shape) > 1:
        y_pred = np.argmax(y_pred, axis=1)

    cm = confusion_matrix(y_true, y_pred)
    n_class = cm.shape[0]
    costs = np.reshape(np.tile(range(n_class), n_class), (n_class, n_class))
    costs = np.abs(costs - np.transpose(costs))
    errors = costs * cm

    # Remove rows with all zeros in the confusion matrix
    non_zero_cm_rows = ~np.all(cm == 0, axis=1)
    errors = errors[non_zero_cm_rows]
    cm = cm[non_zero_cm_rows]

    per_class_maes = np.sum(errors, axis=1) / np.sum(cm, axis=1).astype("double")
    return np.mean(per_class_maes)


def mmae(y_true: np.ndarray, y_pred: np.ndarray):
    """Computes the maximum mean absolute error computed independently for each class
    as presented in :footcite:t:`cruz2014metrics`.

    Parameters
    ----------
    y_true : array-like
            Target labels with one-hot or integer encoding.
    y_pred : array-like
            Predicted probabilities or labels.

    Returns
    -------
    mmae : float
            Maximum mean absolute error.

    Examples
    --------
    >>> import numpy as np
    >>> from dlordinal.metrics import mmae
    >>> y_true = np.array([0, 0, 1, 2, 3, 0, 0])
    >>> y_pred = np.array([0, 1, 1, 2, 3, 0, 1])
    >>> mmae(y_true, y_pred)
    0.5
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    if len(y_true.shape) > 1:
        y_true = np.argmax(y_true, axis=1)
    if len(y_pred.shape) > 1:
        y_pred = np.argmax(y_pred, axis=1)

    cm = confusion_matrix(y_true, y_pred)
    n_class = cm.shape[0]
    costs = np.reshape(np.tile(range(n_class), n_class), (n_class, n_class))
    costs = np.abs(costs - np.transpose(costs))
    errors = costs * cm

    # Remove rows with all zeros in the confusion matrix
    non_zero_cm_rows = ~np.all(cm == 0, axis=1)
    errors = errors[non_zero_cm_rows]
    cm = cm[non_zero_cm_rows]

    per_class_maes = np.sum(errors, axis=1) / np.sum(cm, axis=1).astype("double")
    return per_class_maes.max()


def write_metrics_dict_to_file(
    metrics: Dict[str, float],
    path_str: str,
    filter_fn: Optional[Callable[[str, float], bool]] = lambda n, v: True,
) -> None:
    """Writes a dictionary of metrics to a tabular file.
    The dictionary is filtered by the filter function.
    The first time that the metrics are saved to the file, the keys are written as
    the header. Subsequent calls append the values to the file.

    Parameters
    ----------
    metrics : Dict[str, float]
            Dictionary of metric names associated with their value.
    path_str : str
            Path to the file that will be saved.
            The directory of the file will be created if it does not exist.
            If the file exists, the metrics will be appended to the file in a new row.
    filter_fn : Optional[Callable[[str, bool], bool]], default=lambda n, v: True
            Function that filters the metrics.
            The function takes the name and the value of the metric and returns ``True``
            if the metric should be saved.

    Examples
    --------
    >>> metrics = {'acc': 0.5, 'gmsec': 0.25}
    >>> write_metrics_dict_to_file(metrics, 'results.txt')
    >>> write_metrics_dict_to_file(metrics, 'results.txt')
    >>> with open('results.txt', 'r') as f:
    ...     print(f.read())
    acc	gmsec
    0.5	0.25
    0.5	0.25

    >>> write_metrics_dict_to_file(metrics, 'results.txt', filter_fn=lambda name, value: name == 'acc')
    >>> with open('results.txt', 'r') as f:
    ...     print(f.read())
    acc
    0.5
    0.5
    """

    path = Path(path_str)
    directory = path.parents[0]
    os.makedirs(directory, exist_ok=True)

    if not path.is_file():
        with open(path, "w") as f:
            for k, v in metrics.items():
                if filter_fn(k, v):
                    f.write(f"{k},")
            f.write("\n")

    with open(path, "a") as f:
        for k, v in metrics.items():
            if filter_fn(k, v):
                f.write(f"{v},")
        f.write("\n")


def write_array_to_file(array: np.ndarray, path_str: str, id: str):
    """Writes an array to a json file.
    The array is saved as a dictionary with the key 'id' and the value 'array'.

    Parameters
    ----------
    array : array-like
            Array to be saved.
    path_str : str
            Path to the file that will be saved.
            The directory of the file will be created if it does not exist.
    id : str
            Id of the array.

    Examples
    --------
    >>> array = np.array([0, 1, 2])
    >>> write_array_to_file(array, 'results.json', 'array')
    >>> with open('results.json', 'r') as f:
    ...     print(f.read())
    {"array": [0, 1, 2]}

    >>> array2 = np.array([3, 4, 5])
    >>> write_array_to_file(array, 'results.json', 'array2')
    >>> with open('results.json', 'r') as f:
    ...     print(f.read())
    {"array": [0, 1, 2], "array2": [3, 4, 5]}
    """

    path = Path(path_str)
    directory = path.parents[0]
    os.makedirs(directory, exist_ok=True)

    if path.is_file():
        with open(path, "r") as f:
            data = json.load(f)
    else:
        data = dict()

    data[id] = array.tolist()

    with open(path, "w") as f:
        json.dump(data, f)

In [ ]:
!pip install skorch

In [ ]:
import numpy as np
# from dlordinal.losses import TriangularLoss
# from dlordinal.metrics import amae, mmae
from skorch import NeuralNetClassifier
from torch import nn
from torch.optim import Adam
from torchvision import models
from torchvision.transforms import (Compose,  ToTensor)
import torch
from skorch.callbacks import EpochScoring, EarlyStopping, LRScheduler, Checkpoint
from torchvision.transforms import Compose, Resize, ToTensor
from skorch.helper import predefined_split
from skorch.callbacks import Callback
from torch.utils.data import Subset
import torchvision.transforms as transforms
from skorch.helper import SliceDataset
from albumentations.pytorch import ToTensorV2
from sklearn.utils.class_weight import compute_class_weight
from torchvision.models import efficientnet_b2, EfficientNet_B0_Weights,EfficientNet_B2_Weights

img_width, img_height = 224,224


train_transforms = A.Compose([
    A.RandomResizedCrop(size=(224,224), scale=(0.6, 1.0), p=1.0),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=20, p=0.5),
    A.ColorJitter(brightness=0.3, contrast=0.3, p=0.5),
    A.OneOf([
        # A.MotionBlur(p=0.2),
        # A.MedianBlur(blur_limit=3, p=0.1),
        A.GaussianBlur(p=0.1),
        A.GaussNoise(p=0.2),
    ], p=0.3),
    # A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4),
    A.CoarseDropout(max_height=32, max_width=32, max_holes=3, fill_value=0, p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])


test_transforms = A.Compose([
    A.Resize(height=img_height, width=img_width),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])



DATA_ROOT = "/kaggle/input/r2-new/res2_dataset"
IMAGE_SIZE = (224,224)

train_dataset = CustomImageDataset(
    root=DATA_ROOT,
    train=True,
    target_transform=np.array,
    transform=train_transforms,
)
test_dataset = CustomImageDataset(
    root=DATA_ROOT,
    train=False,
    target_transform=np.array,
    transform=test_transforms
)

num_classes = len(train_dataset.classes)


# ✅ 加入 class weights 計算
y_train = [int(label) for _, label in train_dataset]
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = torch.tensor(class_weights, dtype=torch.float32)


# 模型設定
model = efficientnet_b2(weights=EfficientNet_B2_Weights.IMAGENET1K_V1)


for name, param in model.features.named_parameters():
    param.requires_grad = False  

for name, param in list(model.features.named_parameters())[-15:]:
    param.requires_grad = True
# for name, param in model.features.named_parameters():
#     if '6' in name or '7' in name:
#         param.requires_grad = True
# for param in model.features.parameters():
#     param.requires_grad = True

# # 解凍所有 BatchNorm 層（避免分布偏移），或解凍更多層
# for name, param in model.named_parameters():
#     if  'features.5'in name or'features.6' in name or 'features.7' in name:
#         param.requires_grad = True

    
in_features = model.classifier[1].in_features

model.classifier = nn.Sequential(
    nn.Linear(in_features, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 64),
    # nn.BatchNorm1d(64),
    nn.ReLU(),
    # nn.Dropout(0.2),
    nn.Linear(64, num_classes)
)


weighted_ce_loss = nn.CrossEntropyLoss()
loss_fn = TriangularLoss(base_loss=weighted_ce_loss, num_classes=num_classes, alpha2=0.6 ,eta=0.6)


device = 'cuda' if torch.cuda.is_available() else 'cpu'
true_labels = torch.tensor([int(label) for _, label in test_dataset], dtype=torch.long).to(device)

test_dataset = Subset(test_dataset, range(len(test_dataset)))

estimator = NeuralNetClassifier(
    module=model,
    criterion=loss_fn,
    optimizer=Adam,
    optimizer__weight_decay=5e-4,
    lr=1e-4,
    max_epochs=80,
    batch_size=64,
    verbose=1,
    device='cuda' if torch.cuda.is_available() else 'cpu',  
    train_split=predefined_split(test_dataset),
    callbacks=[
        ('train_acc', EpochScoring('accuracy', on_train=True, name='train_acc')),
        ('lr_scheduler', LRScheduler(policy='ReduceLROnPlateau', mode='min', factor=0.5, patience=10)),
        ('checkpoint', Checkpoint(dirname='checkpoints', f_params='best.pt', monitor='valid_acc_best')),
        # (EarlyStopping(patience=20, monitor="valid_loss", lower_is_better=True)),
    ],
)
y = SliceDataset(train_dataset, idx=1) 
estimator.fit(X=train_dataset, y=y)

history = estimator.history

# 預測
train_probs = estimator.predict_proba(train_dataset)
test_probs = estimator.predict_proba(test_dataset)

# 指標計算
amae_metric = amae(true_labels.cpu().numpy(), test_probs)
mmae_metric = mmae(true_labels.cpu().numpy(), test_probs)

print(f"Test AMAE: {amae_metric:.4f}, Test MMAE: {mmae_metric:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# 提取訓練紀錄資訊
history_data = history.to_list()
epochs = [h['epoch'] for h in history_data]
train_loss = [h['train_loss'] for h in history_data]
valid_loss = [h.get('valid_loss') for h in history_data]
train_acc = [h.get('train_acc') for h in history_data]
valid_acc = [h.get('valid_acc') for h in history_data]

# 畫圖
plt.figure(figsize=(14, 6))

# Loss 曲線
plt.subplot(1, 2, 1)
plt.plot(epochs, train_loss, label='Train Loss')
plt.plot(epochs, valid_loss, label='Valid Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Accuracy 曲線
plt.subplot(1, 2, 2)
plt.plot(epochs, train_acc, label='Train Accuracy')
plt.plot(epochs, valid_acc, label='Valid Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import torch
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from torch import nn
from torch.optim import Adam
from torchvision import models, transforms
from skorch import NeuralNetClassifier
# from dlordinal.losses import TriangularLoss

# 定義與訓練時相同的模型結構
num_classes = 4  # <-- 請改成你的實際類別數

y_train = [int(label) for _, label in train_dataset]
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = torch.tensor(class_weights, dtype=torch.float32)

# 模型設定
model = models.efficientnet_b2(
    weights=None
)


for name, param in model.features.named_parameters():
    param.requires_grad = False  

for name, param in list(model.features.named_parameters())[-15:]:
    param.requires_grad = True

in_features = model.classifier[1].in_features

model.classifier = nn.Sequential(
    nn.Linear(in_features, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 64),
    # nn.BatchNorm1d(64),
    nn.ReLU(),
    # nn.Dropout(0.2),
    nn.Linear(64, num_classes)
)
# 同樣的 loss 函數

weighted_ce_loss = nn.CrossEntropyLoss(weight=class_weights)
loss_fn = TriangularLoss(base_loss=weighted_ce_loss, num_classes=num_classes, alpha2=0.6 ,eta=0.6)
# 初始化 skorch 模型
inference_estimator = NeuralNetClassifier(
    module=model,
    criterion=loss_fn,
    optimizer=Adam,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

# 初始化並載入參數
inference_estimator.initialize()
inference_estimator.load_params(f_params='checkpoints/best.pt')  # 你的儲存位置

# test_dataset 應該已經被你定義過了，如果是 Subset 物件，取得真實標籤：
y_true = np.array([test_dataset.dataset[i][1] for i in test_dataset.indices])

# 使用載入的最佳模型做預測
y_pred = inference_estimator.predict(test_dataset)

# 類別名稱（可自訂）
target_names = [str(i) for i in range(num_classes)]

# Classification Report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=target_names))

# 混淆矩陣
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
